In [1]:
import numpy as np
import pandas as pd
import os
import kagglehub

In [2]:
%cd /kaggle/working/ 
!git clone https://github.com/HLE1236/airace --recursive

/kaggle/working
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 941, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 941 (delta 49), reused 56 (delta 25), pack-reused 858 (from 1)
Receiving objects: 100% (941/941), 78.70 MiB | 38.30 MiB/s, done.
Resolving deltas: 100% (518/518), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/kaggle/working/gaussian-splatting/SIBR_viewers'...
remote: Enumerating object

In [3]:
%cd /kaggle/working/gaussian-splatting
!pip install submodules/diff-gaussian-rasterization
!pip install submodules/simple-knn
!pip install submodules/fused-ssim
!pip install plyfile 
!pip install pycolmap
!pip install lpips

/kaggle/working/gaussian-splatting
Processing ./submodules/diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp312-cp312-linux_x86_64.whl size=3812395 sha256=1f6c8e4b8859c2c7defa85fa340a7f25f0c8bb7e5585c1abefdfaf6621415814
  Stored in directory: /root/.cache/pip/wheels/ba/99/d3/014520068aca8c2e8bdc358ca774581380cadb65788559b3ea
Successfully built diff_gaussian_rasterization
Processing ./submodules/simple-knn
  Preparing metadata (setup.py) ... done
  Created wheel for simple_knn: filename=simple_knn-0.0.0-cp312-cp312-linux_x86_64.whl size=3552528 sha256=c38ab1dd8bdfc344e9e6b05e362f6d3a7603d1acf3430e580298064796b3bce8
  Stored in directory: /root/.cache/pip/wheels/ca/30/df/7f4f362d12edead48c699acde5962cbb06ca05033b9d970934
Successfully built simple_knn
Processing ./submodules/fused-ssim
  Preparing metadata (setup.py) ... done
  Created wheel for fused_ssim: filename=fused_ssim

In [4]:
!apt-get update && apt-get install -y colmap

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,812 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy

In [5]:
!python preprocess.py --input "/kaggle/input/datasets/xuanph/phase1/phase1/private_set1" --output "/kaggle/working/cleaned_inputs" --subset HCM0249

Undistorting HCM0249 (COLMAP CLI)...
Generating alpha masks (COLMAP CLI on white images)...
/kaggle/working/gaussian-splatting/preprocess.py:152: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(rgba, mode="RGBA").save(out_path)
Đã ghi alpha channel vào 240 ảnh undistort tại /kaggle/working/cleaned_inputs/HCM0249/train/images
  Đã đồng bộ tên 240 ảnh trong images.bin (đổi đuôi sau embed alpha)
scene: HCM0249
Tổng ảnh: 306
Số ảnh missing: 66
Còn lại sau khi xóa: 240
Đã cập nhật reconstruction.


In [6]:
import subprocess
import sys
import os

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    sys.executable,
    "train_full.py",
    "--subset", "HCM0249",
    "--cap_max", "5850000",
    "--iterations", "30000"
]

ret = subprocess.run(cmd, env=env)

if ret.returncode != 0:
    print(f"!!! Training failed with code {ret.returncode}")
else:
    print("Training completed successfully.")

Training scene: HCM0249
  source_path = /kaggle/working/cleaned_inputs/HCM0249/train
  model_path  = /kaggle/working/model_outputs/HCM0249
Output folder: /kaggle/working/model_outputs/HCM0249
Reading camera 240/240
Converting point3d.bin to .ply, will happen only the first time you open the scene.
Loading Training Cameras
Loading Test Cameras
Number of points at initialisation :  165726


Training progress: 100%|██████████| 30000/30000 [2:52:00<00:00,  2.91it/s, Loss=0.0401578, Depth Loss=0.0000000, N=5647244]



[ITER 7000] Evaluating train: L1 0.04479419887065888 PSNR 21.507529449462893

[ITER 7000] Saving Gaussians

[ITER 30000] Evaluating train: L1 0.03218509405851364 PSNR 22.87689437866211

[ITER 30000] Saving Gaussians
Found 8 scenes: ['HCM0249', 'HCM0254', 'HCM0276', 'HCM1439', 'HNI0131', 'HNI0265', 'HNI0366', 'HNI0437']
Filtered to subset (1): ['HCM0249']

=== [1/1] Training scene: HCM0249 ===

Done. 1/1 scenes succeeded.
Training completed successfully.


In [7]:
%cd /kaggle/working/gaussian-splatting
!python render_full.py --iterations 30000 --subset HCM0249

/kaggle/working/gaussian-splatting
Found 8 scenes: ['HCM0249', 'HCM0254', 'HCM0276', 'HCM1439', 'HNI0131', 'HNI0265', 'HNI0366', 'HNI0437']
Filtered to subset (1): ['HCM0249']

=== [1/1] Rendering scene: HCM0249 ===
Looking for config file in /kaggle/working/model_outputs/HCM0249/cfg_args
Config file found: /kaggle/working/model_outputs/HCM0249/cfg_args
Loading trained model at iteration 30000 [12/07 05:12:53]
[HCM0249] distortion k=0.008910 f=928.20 cx=660.00 cy=494.50 size=(1320x989) [12/07 05:13:04]
[HCM0249] undistorted render canvas f=928.20 cx=660.00 cy=494.50 size=(1320x989) [12/07 05:13:04]
Rendering HCM0249: 100%|████████████████████████| 60/60 [00:10<00:00,  5.97it/s]
Rendered 60 images for HCM0249 from iteration 30000 -> /kaggle/working/image_outputs/HCM0249 [12/07 05:13:14]

Done. 1/1 scenes succeeded.


In [8]:
from pathlib import Path
import shutil

subset = ["HCM0249"]
root = Path("/kaggle/working")

for scene in subset:
    model_dir = root / "model_outputs" / scene
    if model_dir.exists():
        shutil.make_archive(
            base_name=str(root / f"{scene}_model"),
            format="zip",
            root_dir=model_dir.parent,
            base_dir=model_dir.name,
        )

    image_dir = root / "image_outputs" / scene
    if image_dir.exists():
        shutil.make_archive(
            base_name=str(root / f"{scene}_image"),
            format="zip",
            root_dir=image_dir.parent,
            base_dir=image_dir.name,
        )

print("Done!")

Done!
